
# Diachronic Sentiment Arcs — 2025 Teaching Notebook (Clean & Reproducible)

**Audience:** non‑STEM students and newcomers to text mining  
**Goal:** turn long text into a *time series of affect* (“sentiment arc”), compare three sentiment families, smooth responsibly (moving avg / EMA / **Savitzky–Golay**), detect peaks/valleys, and discuss uncertainty.

### Reading the roadmap
1. Setup (reproducible environment + versions)  
2. Load or paste text → sentence segmentation  
3. Sentiment families: **Lexicon (VADER)**, **Classic ML baseline**, **Transformers (DistilBERT)**  
4. From sentences → time series (scores and confidence)  
5. Smoothing: MA, EMA, **Savitzky–Golay** (when/why)  
6. Peaks/valleys + interpretation caveats  
7. Robustness: model agreement, bootstrap bands  
8. Ethics & limitations  
9. **R Appendix** (optional): syuzhet/sentimentr, if your runtime supports R/rpy2

> **Pedagogical reminder:** these curves are *descriptive*. Peaks suggest clusters of positive sentences; valleys, negative. They are not causal proof of emotions or author intent.



## 1) Setup & Environment

- Run the install cell **once at the top** (avoids state drift).
- Then capture versions for reproducibility.


In [ ]:
%%bash
# Improved installation with better error handling and version pinning
echo "Installing dependencies..."

# Install core packages first
pip install -q "torch>=2.0" "numpy>=1.24" "pandas>=2.0" --upgrade

# Install ML packages  
pip install -q "transformers>=4.44,<5" "datasets>=3.0,<3.1" "accelerate>=1.0" --upgrade

# Install text processing and sentiment packages
pip install -q "vaderSentiment>=3.3.2" "textblob>=0.18" "pysbd>=0.3" --upgrade

# Install visualization and analysis packages
pip install -q "matplotlib>=3.9" "plotly>=5.24" "scipy>=1.13" "scikit-learn>=1.5" --upgrade

# Install utility packages
pip install -q "tqdm>=4.66" --upgrade

echo "Downloading NLTK data..."
python - <<'PY'
import nltk
import sys
try:
    nltk.download("punkt", quiet=True)
    nltk.download("vader_lexicon", quiet=True)
    print("NLTK data downloaded successfully")
except Exception as e:
    print(f"Warning: NLTK download failed: {e}")
    print("You may need to download NLTK data manually")
PY

echo "Installation complete!"

In [ ]:
# Capture versions (keep outputs in your notebook for reproducibility)
import sys, platform, importlib

print("Python:", sys.version)
print("Platform:", platform.platform())

# Version checking with better error handling
modules_to_check = [
    "torch", "transformers", "datasets", "pandas", "numpy", 
    "matplotlib", "scipy", "sklearn", "vaderSentiment", "pysbd", "plotly", "nltk"
]

print("\nLibrary Versions:")
print("=" * 20)
for mod in modules_to_check:
    try:
        m = importlib.import_module(mod)
        version = getattr(m, "__version__", "n/a")
        print(f"{mod}: {version}")
    except Exception as e:
        print(f"{mod}: not installed ({e})")

# Check NLTK data availability
try:
    import nltk
    print(f"\nNLTK version: {nltk.__version__}")
    
    # Check if required data is available
    try:
        nltk.data.find('tokenizers/punkt')
        print("✓ NLTK punkt data available")
    except LookupError:
        print("⚠ NLTK punkt data not available - downloading...")
        nltk.download("punkt", quiet=True)
    
    try:
        nltk.data.find('sentiment/vader_lexicon.zip')
        print("✓ NLTK vader_lexicon available")
    except LookupError:
        print("⚠ NLTK vader_lexicon not available - downloading...")
        nltk.download("vader_lexicon", quiet=True)
        
except Exception as e:
    print(f"NLTK setup issue: {e}")

# Check device availability
try:
    import torch
    print(f"\nPyTorch version: {torch.__version__}")
    
    if torch.cuda.is_available():
        print(f"✓ CUDA available: {torch.cuda.device_count()} device(s)")
        print(f"  Current device: {torch.cuda.get_device_name()}")
    else:
        print("⚠ CUDA not available")
        
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        print("✓ Apple Silicon MPS available")
    else:
        print("⚠ Apple Silicon MPS not available")
        
except Exception as e:
    print(f"PyTorch device check failed: {e}")

print(f"\nEnvironment check complete.")


## 2) Configuration

Centralized parameters so we can tweak behavior without editing many cells.


In [ ]:

from dataclasses import dataclass
from pathlib import Path
import numpy as np, random
import re

@dataclass
class Config:
    project_name: str = "SentimentArcs"
    seed: int = 7
    # Transformer classifier (binary SST-2 is good for pedagogy)
    model_name: str = "distilbert-base-uncased-finetuned-sst-2-english"
    batch_size: int = 32
    max_length: int = 256
    # Smoothing choices
    smoothing: str = "savgol"    # options: "ma","ema","savgol"
    window_frac: float = 0.10    # ~10% of series length
    savgol_poly: int = 3         # poly order for Savitzky–Golay
    ema_alpha: float = 0.2       # EMA responsiveness (0<alpha<=1)
    # Peak detection
    peak_min_frac: float = 0.05  # min distance as % of series length
    # Plotting
    interactive: bool = True     # also render Plotly charts

CFG = Config()

# Seeding
np.random.seed(CFG.seed)
random.seed(CFG.seed)
try:
    import torch
    torch.manual_seed(CFG.seed)
except Exception:
    pass

out_dir = Path("./outputs"); out_dir.mkdir(exist_ok=True, parents=True)



## 3) Load or Paste Your Text → Sentence Segmentation

Paste text in the cell below **or** mount Drive and read from a file. We prefer clear, idempotent functions for cleaning/segmenting.


In [ ]:

SAMPLE_TEXT = '''
Alice felt cheerful on Monday. The sun was bright, and the city buzzed.
On Tuesday, a storm hit; plans were canceled, and frustration rose.
Wednesday brought relief as friends visited. Thursday was neutral.
By Friday, optimism returned in full force!
'''


In [ ]:

def clean_text(txt: str) -> str:
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt

def segment_sentences(txt: str):
    try:
        import pysbd
        seg = pysbd.Segmenter(language="en", clean=True)
        sents = [s.strip() for s in seg.segment(txt)]
    except Exception:
        # very simple fallback: split on punctuation
        sents = re.split(r"(?<=[.!?])\s+", txt)
    return [s for s in sents if s]


In [ ]:

import pandas as pd
text = clean_text(SAMPLE_TEXT)
sentences = segment_sentences(text)
df = pd.DataFrame({"i": range(len(sentences)), "sentence": sentences})
df.head()



## 4) Three Sentiment Families (Compare & Contrast)

- **Lexicon (VADER)**: rules + valence shifters. Fast, transparent. Domain slang may be tricky.  
- **Classic ML baseline (TF‑IDF + Logistic Regression)**: requires labels to train; we include a scaffold.  
- **Transformers (DistilBERT)**: contextual; strong accuracy; heavier. We use `pipeline(..., batch_size=CFG.batch_size)` with explicit model choice to keep results stable.


### 4.1 Lexicon: VADER (no training needed)

In [ ]:

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

def vader_score(sent: str) -> float:
    # return compound in [-1,1]
    return float(sia.polarity_scores(sent)["compound"])

df["vader"] = [vader_score(s) for s in df["sentence"]]
df.head()



### 4.2 Classic ML Baseline (optional, needs labels)

If you have labeled data, train TF‑IDF + Logistic Regression. Here we only show a scaffold.


In [ ]:

# Scaffold (no training without labels)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

baseline = make_pipeline(
    TfidfVectorizer(max_features=20000, ngram_range=(1,2)),
    LogisticRegression(max_iter=1000)
)
# Example (skip actual fit):
# baseline.fit(train_texts, train_labels)
# proba = baseline.predict_proba(df["sentence"])[:, 1]  # map to [-1,1] later


### 4.3 Transformers: DistilBERT sentiment (batched & explicit)

In [ ]:
from transformers import pipeline
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore", category=FutureWarning)

# Determine device automatically with fallback
device = -1  # CPU fallback
try:
    import torch
    if torch.cuda.is_available():
        device = 0  # GPU
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = "mps"  # Apple Silicon GPU
except Exception:
    pass

print(f"Using device: {'GPU' if device >= 0 else 'CPU'}")

clf = pipeline(
    task="sentiment-analysis",
    model=CFG.model_name,
    tokenizer=CFG.model_name,
    truncation=True,
    max_length=CFG.max_length,
    device=device  # Use explicit device instead of device_map
)

# batched inference with error handling
try:
    preds = clf(df["sentence"].tolist(), batch_size=CFG.batch_size, padding=True, truncation=True)
except Exception as e:
    print(f"Error in batched inference: {e}")
    print("Falling back to individual sentence processing...")
    preds = clf(df["sentence"].tolist())

# Convert labels to signed scores multiplied by confidence
label_to_sign = {"NEGATIVE": -1, "POSITIVE": +1, "NEUTRAL": 0}
def signed(p):
    label = p.get("label", "").upper()
    score = float(p.get("score", 1.0))
    return label_to_sign.get(label, 0) * score

df["hf_score"] = [signed(p) for p in preds]
df.head()


## 5) From Sentences → Time Series + Smoothing

We get one score per sentence, then smooth.  
**Smoothing choices:** Moving Average (`window`), Exponential MA (`alpha`), **Savitzky–Golay** (`window_length` *odd*, `polyorder < window_length`).  
*Rule of thumb:* start with a window covering **5–15%** of points.


In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")

def z(x: np.ndarray) -> np.ndarray:
    """Standardize with improved NaN handling"""
    x = np.asarray(x, dtype=float)
    mean_val = np.nanmean(x)
    std_val = np.nanstd(x)
    if std_val < 1e-8:  # Handle zero variance case
        warnings.warn("Variance is near zero, returning zeros")
        return np.zeros_like(x)
    return (x - mean_val) / std_val

def moving_average(x, k):
    """Moving average with improved edge handling"""
    k = max(1, int(k))
    if k >= len(x):
        warnings.warn(f"Window size {k} >= data length {len(x)}, using simple mean")
        return np.full_like(x, np.mean(x))
    return np.convolve(x, np.ones(k) / k, mode="same")

def exponential_moving_average(x, alpha):
    """Exponential moving average with input validation"""
    if not 0 < alpha <= 1:
        raise ValueError(f"Alpha must be in (0,1], got {alpha}")
    
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)
    y[0] = x[0]
    for t in range(1, len(x)):
        y[t] = alpha * x[t] + (1 - alpha) * y[t-1]
    return y

def savgol_smooth(x, window_length, polyorder):
    """Savitzky-Golay smoothing with better error handling"""
    try:
        from scipy.signal import savgol_filter
        x = np.asarray(x, dtype=float)
        
        # Validate inputs
        if len(x) < 3:
            warnings.warn("Too few points for Savitzky-Golay, using moving average")
            return moving_average(x, min(len(x), 3))
        
        wl = max(3, int(window_length) | 1)  # ensure odd & >=3
        wl = min(wl, len(x))  # don't exceed data length
        poly = min(polyorder, wl-1)
        poly = max(1, poly)  # ensure at least linear
        
        return savgol_filter(x, wl, poly, mode="interp")
    except Exception as e:
        warnings.warn(f"Savitzky-Golay failed ({e}), falling back to moving average")
        return moving_average(x, max(3, int(window_length)))

def smooth_series(y, method="savgol", window_frac=0.1, poly=3, alpha=0.2):
    """Main smoothing function with comprehensive validation"""
    y = np.asarray(y, float)
    n = len(y)
    
    if n == 0:
        raise ValueError("Input series is empty")
    if n == 1:
        return y.copy()
    
    # Calculate window size
    k = max(3, int(np.ceil(n*window_frac)) | 1)  # odd
    k = min(k, n)  # don't exceed data length
    
    if method == "ma":
        return moving_average(y, k)
    elif method == "ema":
        return exponential_moving_average(y, alpha)
    elif method == "savgol":
        return savgol_smooth(y, k, poly)
    else:
        raise ValueError(f"Unknown smoothing method: {method}")

# Prepare series with error handling
try:
    df["vader_z"] = z(df["vader"].values)
    df["hf_z"] = z(df["hf_score"].values)
    
    # Fix the statistical calculation error - use axis=0 for correct column-wise mean
    vader_z_array = df["vader_z"].values
    hf_z_array = df["hf_z"].values
    
    # Stack properly: shape (2, n) not (n, 2)
    stacked_scores = np.vstack([vader_z_array, hf_z_array])
    df["mean_z"] = z(np.mean(stacked_scores, axis=0))
    
    df["mean_z_smooth"] = smooth_series(df["mean_z"].values, method=CFG.smoothing,
                                        window_frac=CFG.window_frac, poly=CFG.savgol_poly,
                                        alpha=CFG.ema_alpha)
    
    print(f"Successfully processed {len(df)} sentences")
    print(f"Smoothing method: {CFG.smoothing}, window fraction: {CFG.window_frac}")
    
except Exception as e:
    print(f"Error in time series processing: {e}")
    raise

df[["i","sentence","vader_z","hf_z","mean_z","mean_z_smooth"]].head()


## 6) Visualizing the Arc (static + optional interactive)

- **Static** first for quick reading.  
- **Interactive** (Plotly) adds hover text and easier peak inspection.


In [ ]:

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10,4))
ax.plot(df["i"], df["mean_z"], label="mean z", linewidth=1)
ax.plot(df["i"], df["mean_z_smooth"], label=f"smooth ({CFG.smoothing})", linewidth=2)
ax.set_xlabel("Sentence #"); ax.set_ylabel("Sentiment (z-score)")
ax.set_title("Diachronic Sentiment Arc")
ax.legend(loc="best")
plt.show()


In [ ]:

if CFG.interactive:
    import plotly.graph_objects as go
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["i"], y=df["mean_z"], mode="lines", name="mean z",
                             hovertext=df["sentence"], hoverinfo="text+x+y"))
    fig.add_trace(go.Scatter(x=df["i"], y=df["mean_z_smooth"], mode="lines", name=f"smooth ({CFG.smoothing})"))
    fig.update_layout(title="Diachronic Sentiment Arc (Interactive)",
                      xaxis_title="Sentence #", yaxis_title="Sentiment (z-score)")
    fig.show()



## 7) Peaks/Valleys + Uncertainty

- We detect peaks on the **smoothed** series with a minimum spacing (e.g., 5% of the length).  
- We bootstrap (resample sentences with replacement) to estimate an uncertainty band (±1 SD) around the curve.


In [ ]:
from math import ceil
import numpy as np
import warnings

try:
    from scipy.signal import find_peaks
    print("Using scipy.signal.find_peaks")
except Exception:
    warnings.warn("scipy.signal.find_peaks not available, using fallback implementation")
    def find_peaks(x, distance=1):
        """Fallback peak detection - simple local maxima"""
        x = np.asarray(x)
        if len(x) < 3:
            return np.array([]), {}
        
        peaks = [i for i in range(1, len(x)-1) if x[i] > x[i-1] and x[i] > x[i+1]]
        
        # Filter by distance
        if distance > 1:
            filtered_peaks = []
            for p in peaks:
                if not filtered_peaks or p - filtered_peaks[-1] >= distance:
                    filtered_peaks.append(p)
            peaks = filtered_peaks
            
        return np.array(peaks), {}

# Peak detection with better error handling
try:
    if len(df) < 3:
        warnings.warn("Too few sentences for reliable peak detection")
        df_peaks = pd.DataFrame(columns=["i", "sentence", "mean_z_smooth"])
    else:
        # Calculate minimum distance between peaks
        dist = max(1, int(ceil(len(df) * CFG.peak_min_frac)))
        print(f"Minimum peak distance: {dist} sentences")
        
        # Find peaks on smoothed series
        peaks, properties = find_peaks(df["mean_z_smooth"].values, distance=dist)
        
        if len(peaks) == 0:
            warnings.warn("No peaks found in the sentiment arc")
            df_peaks = pd.DataFrame(columns=["i", "sentence", "mean_z_smooth"])
        else:
            df_peaks = df.loc[peaks, ["i", "sentence", "mean_z_smooth"]].copy()
            print(f"Found {len(peaks)} peaks at sentences: {peaks.tolist()}")
        
except Exception as e:
    warnings.warn(f"Peak detection failed: {e}")
    df_peaks = pd.DataFrame(columns=["i", "sentence", "mean_z_smooth"])

df_peaks

In [ ]:
# Bootstrap uncertainty band with improved error handling
import numpy as np
import warnings
from tqdm.auto import tqdm

print("Running bootstrap analysis...")

# Validate input
if len(df) == 0:
    raise ValueError("DataFrame is empty - cannot run bootstrap analysis")

B = 200  # bootstrap samples (keep small for class runtimes)
vals = df["mean_z"].values

if len(vals) < 3:
    warnings.warn("Too few sentences for meaningful bootstrap analysis")
    B = 50  # Reduce samples for very short texts

curves = []
print(f"Generating {B} bootstrap samples...")

try:
    for b in tqdm(range(B), desc="Bootstrap progress"):
        # Resample with replacement
        idx = np.random.RandomState(CFG.seed + b).choice(len(vals), size=len(vals), replace=True)
        series = vals[idx]
        
        # Apply smoothing with error handling
        try:
            sm = smooth_series(series, method=CFG.smoothing, window_frac=CFG.window_frac,
                             poly=CFG.savgol_poly, alpha=CFG.ema_alpha)
            curves.append(sm)
        except Exception as e:
            warnings.warn(f"Bootstrap sample {b} failed: {e}")
            # Use original smoothing as fallback
            sm = df["mean_z_smooth"].values
            curves.append(sm)
    
    if len(curves) == 0:
        raise RuntimeError("All bootstrap samples failed")
    
    curves = np.vstack(curves)
    mean = curves.mean(axis=0)
    sd = curves.std(axis=0)
    
    # Add to dataframe
    df["smooth_mean"] = mean
    df["smooth_sd"] = sd
    
    print(f"Bootstrap analysis complete. Mean SD: {np.mean(sd):.3f}")
    
    # Check for reasonable uncertainty bounds
    avg_sd = np.mean(sd)
    if avg_sd > 2.0:
        warnings.warn(f"Large uncertainty detected (avg SD: {avg_sd:.2f}) - results may be unreliable")
    elif avg_sd < 0.1:
        warnings.warn(f"Very small uncertainty detected (avg SD: {avg_sd:.3f}) - may indicate over-smoothing")

except Exception as e:
    print(f"Error in bootstrap analysis: {e}")
    # Fallback: use simple empirical estimates
    df["smooth_mean"] = df["mean_z_smooth"].values
    df["smooth_sd"] = np.full(len(df), 0.5)  # Conservative uncertainty estimate
    warnings.warn("Using fallback uncertainty estimates")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("Creating uncertainty band visualization...")

# Validate data
if len(df) == 0:
    raise ValueError("No data to plot")

if "smooth_mean" not in df.columns or "smooth_sd" not in df.columns:
    raise ValueError("Bootstrap results not found - run bootstrap analysis first")

x = df["i"].values

# Create figure with better layout
fig, ax = plt.subplots(figsize=(12, 5))

# Plot main smoothed curve
ax.plot(x, df["mean_z_smooth"], label="smoothed sentiment", linewidth=2.5, color='blue', alpha=0.8)

# Plot uncertainty band with better aesthetics
ax.fill_between(x, df["smooth_mean"]-df["smooth_sd"], df["smooth_mean"]+df["smooth_sd"],
                alpha=0.25, color='lightblue', label="bootstrap ±1 SD")

# Plot original data points
ax.scatter(x, df["mean_z"], alpha=0.6, s=20, color='gray', label="original points", zorder=5)

# Plot peaks if any exist
if len(df_peaks) > 0:
    ax.scatter(df_peaks["i"], df_peaks["mean_z_smooth"], 
               marker="x", s=80, color='red', label=f"peaks ({len(df_peaks)})", 
               linewidth=2, zorder=10)
    print(f"Plotted {len(df_peaks)} peaks")
else:
    print("No peaks to plot")

# Add zero line for reference
ax.axhline(y=0, color='black', linestyle='--', alpha=0.3, linewidth=1)

# Improve labeling and formatting
ax.set_title("Sentiment Arc with Uncertainty Bands", fontsize=14, fontweight='bold')
ax.set_xlabel("Sentence Number", fontsize=12)
ax.set_ylabel("Sentiment (z-score)", fontsize=12)
ax.legend(loc="best", fontsize=10)
ax.grid(True, alpha=0.3)

# Set reasonable y-limits
y_margin = 0.5
y_min = min(df["mean_z_smooth"].min(), df["smooth_mean"].min() - df["smooth_sd"].max()) - y_margin
y_max = max(df["mean_z_smooth"].max(), df["smooth_mean"].max() + df["smooth_sd"].max()) + y_margin
ax.set_ylim(y_min, y_max)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nSummary Statistics:")
print(f"- Sentences analyzed: {len(df)}")
print(f"- Average sentiment: {df['mean_z'].mean():.3f}")
print(f"- Sentiment range: [{df['mean_z'].min():.3f}, {df['mean_z'].max():.3f}]")
print(f"- Average uncertainty: {df['smooth_sd'].mean():.3f}")
print(f"- Peaks detected: {len(df_peaks)}")


## 8) Model Agreement (Correlation) & Neutral Share

We compare normalized series and report simple statistics to caution against over‑interpreting noisy arcs.


In [ ]:
print("Analyzing model agreement and sentiment distribution...")

# Validate required columns exist
required_cols = ["vader_z", "hf_z", "mean_z"]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

try:
    # Model agreement analysis
    corr = df[["vader_z", "hf_z", "mean_z"]].corr(method="spearman")
    print("Model Agreement (Spearman Correlation):")
    print("=" * 40)
    print(corr.round(3))
    print()
    
    # Individual correlation coefficients
    vader_hf_corr = df["vader_z"].corr(df["hf_z"], method="spearman")
    print(f"VADER ↔ Transformer correlation: {vader_hf_corr:.3f}")
    
    # Interpret correlation strength
    if abs(vader_hf_corr) >= 0.7:
        agreement_level = "strong"
    elif abs(vader_hf_corr) >= 0.4:
        agreement_level = "moderate"
    else:
        agreement_level = "weak"
    print(f"Agreement level: {agreement_level}")
    
except Exception as e:
    print(f"Error computing correlations: {e}")
    print("Skipping correlation analysis")

# Sentiment distribution analysis
try:
    # Neutral share calculation
    neutral_threshold = 0.25
    neutral_share = float((np.abs(df["mean_z"]) < neutral_threshold).mean())
    print(f"\nSentiment Distribution:")
    print("=" * 25)
    print(f"Neutral-ish share (|z| < {neutral_threshold}): {neutral_share:.1%}")
    
    # Positive and negative shares
    positive_share = float((df["mean_z"] >= neutral_threshold).mean())
    negative_share = float((df["mean_z"] <= -neutral_threshold).mean())
    print(f"Positive share (z ≥ {neutral_threshold}): {positive_share:.1%}")
    print(f"Negative share (z ≤ -{neutral_threshold}): {negative_share:.1%}")
    
    # Extreme sentiment analysis
    extreme_threshold = 1.0
    extreme_positive = float((df["mean_z"] >= extreme_threshold).mean())
    extreme_negative = float((df["mean_z"] <= -extreme_threshold).mean())
    print(f"\nExtreme sentiment (|z| ≥ {extreme_threshold}):")
    print(f"  Very positive: {extreme_positive:.1%}")
    print(f"  Very negative: {extreme_negative:.1%}")
    
    # Descriptive statistics
    print(f"\nDescriptive Statistics (mean_z):")
    print("=" * 35)
    print(f"  Mean: {df['mean_z'].mean():.3f}")
    print(f"  Std Dev: {df['mean_z'].std():.3f}")
    print(f"  Min: {df['mean_z'].min():.3f}")
    print(f"  Max: {df['mean_z'].max():.3f}")
    print(f"  Range: {df['mean_z'].max() - df['mean_z'].min():.3f}")
    
except Exception as e:
    print(f"Error in sentiment distribution analysis: {e}")

# Model-specific statistics
print(f"\nModel-Specific Statistics:")
print("=" * 30)
try:
    print(f"VADER - Mean: {df['vader_z'].mean():.3f}, Std: {df['vader_z'].std():.3f}")
    print(f"Transformer - Mean: {df['hf_z'].mean():.3f}, Std: {df['hf_z'].std():.3f}")
    
    # Check for systematic differences
    mean_diff = df['vader_z'].mean() - df['hf_z'].mean()
    print(f"Mean difference (VADER - Transformer): {mean_diff:.3f}")
    
    if abs(mean_diff) > 0.2:
        print("⚠️  Large systematic difference between models detected")
    elif abs(mean_diff) > 0.1:
        print("⚡ Moderate systematic difference between models")
    else:
        print("✓ Good agreement between models")
        
except Exception as e:
    print(f"Error in model-specific analysis: {e}")

# Quality indicators
print(f"\nData Quality Indicators:")
print("=" * 25)
print(f"Total sentences: {len(df)}")
print(f"Missing values: {df.isnull().sum().sum()}")

# Check for potential issues
if len(df) < 10:
    print("⚠️  Very short text - results may be unreliable")
elif len(df) < 30:
    print("⚡ Short text - interpret with caution")
else:
    print("✓ Adequate text length for analysis")

if "smooth_sd" in df.columns:
    avg_uncertainty = df["smooth_sd"].mean()
    if avg_uncertainty > 1.0:
        print("⚠️  High uncertainty in smoothing")
    elif avg_uncertainty < 0.2:
        print("⚡ Very low uncertainty - possibly over-smoothed")
    else:
        print("✓ Reasonable uncertainty levels")


## 9) Ethics & Limitations

- **Domain shift:** models trained on movie/product reviews may misread literature, news, or historical texts.  
- **Irony/sarcasm:** still hard for machines.  
- **Cultural nuance:** lexicons and labels embed biases.  
- **Over-smoothing:** can erase minority voices/events; **under-smoothing** overreacts to single sentences.  
- **Do not claim causality** from curve shapes.



---
# R Appendix (Optional)

Use only if your environment supports `rpy2` and R packages; otherwise skip.


In [ ]:

USE_R = False  # set True if you have R + rpy2 + packages installed


In [ ]:

if USE_R:
    # In Colab you can enable R magic by installing rpy2, then:
    # %load_ext rpy2.ipython
    # %%R
    # install.packages(c("sentimentr","syuzhet"))
    # library(sentimentr); library(syuzhet)
    # # ... run R-based arcs here ...
    pass


# Comprehensive Testing and Validation
print("=" * 60)
print("COMPREHENSIVE NOTEBOOK VALIDATION")
print("=" * 60)

import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

def test_section(section_name, test_func):
    """Helper function to run tests and report results"""
    try:
        result = test_func()
        print(f"✓ {section_name}: PASSED")
        return True
    except Exception as e:
        print(f"✗ {section_name}: FAILED - {e}")
        return False

# Test 1: Basic imports and setup
def test_imports():
    required_vars = ['CFG', 'df', 'sia', 'clf']
    for var in required_vars:
        if var not in globals():
            raise ValueError(f"Required variable '{var}' not found")
    return True

# Test 2: Data structure validation
def test_data_structure():
    if df.empty:
        raise ValueError("DataFrame is empty")
    
    required_columns = ['i', 'sentence', 'vader', 'hf_score', 'vader_z', 'hf_z', 'mean_z', 'mean_z_smooth']
    missing_cols = [col for col in required_columns if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns: {missing_cols}")
    
    # Check for NaN values in critical columns
    nan_cols = ['vader_z', 'hf_z', 'mean_z', 'mean_z_smooth']
    for col in nan_cols:
        if df[col].isnull().any():
            raise ValueError(f"NaN values found in {col}")
    
    return True

# Test 3: Sentiment score validation
def test_sentiment_scores():
    # VADER scores should be in [-1, 1]
    if not df['vader'].between(-1, 1).all():
        raise ValueError("VADER scores outside [-1, 1] range")
    
    # z-scores should be roughly standardized
    for col in ['vader_z', 'hf_z', 'mean_z']:
        mean_val = df[col].mean()
        std_val = df[col].std()
        if abs(mean_val) > 0.1:  # Should be close to 0
            raise ValueError(f"{col} not properly standardized (mean={mean_val:.3f})")
        if abs(std_val - 1.0) > 0.1:  # Should be close to 1
            raise ValueError(f"{col} not properly standardized (std={std_val:.3f})")
    
    return True

# Test 4: Smoothing algorithm validation
def test_smoothing():
    # Smoothed series should be less noisy than original
    original_var = df['mean_z'].var()
    smoothed_var = df['mean_z_smooth'].var()
    
    if smoothed_var > original_var * 1.1:  # Allow small increase due to edge effects
        raise ValueError("Smoothing increased variance significantly")
    
    # Check that smoothing preserved the general trend
    correlation = df['mean_z'].corr(df['mean_z_smooth'])
    if correlation < 0.7:
        raise ValueError(f"Smoothing destroyed original pattern (correlation={correlation:.3f})")
    
    return True

# Test 5: Peak detection validation
def test_peak_detection():
    if 'df_peaks' not in globals():
        raise ValueError("Peak detection results not found")
    
    # Peaks should be valid indices
    if len(df_peaks) > 0:
        if not df_peaks['i'].between(0, len(df)-1).all():
            raise ValueError("Peak indices out of range")
        
        # Peaks should actually be local maxima
        for _, row in df_peaks.iterrows():
            i = int(row['i'])
            if i > 0 and i < len(df) - 1:
                current = row['mean_z_smooth']
                prev_val = df.iloc[i-1]['mean_z_smooth']
                next_val = df.iloc[i+1]['mean_z_smooth']
                if current <= max(prev_val, next_val):
                    raise ValueError(f"Index {i} is not a true peak")
    
    return True

# Test 6: Bootstrap validation
def test_bootstrap():
    required_bootstrap_cols = ['smooth_mean', 'smooth_sd']
    missing_cols = [col for col in required_bootstrap_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Bootstrap results missing: {missing_cols}")
    
    # Standard deviation should be positive
    if (df['smooth_sd'] <= 0).any():
        raise ValueError("Non-positive standard deviation values found")
    
    # Uncertainty should be reasonable (not too large or small)
    avg_sd = df['smooth_sd'].mean()
    if avg_sd < 0.05:
        raise ValueError(f"Uncertainty too small (avg_sd={avg_sd:.3f})")
    if avg_sd > 3.0:
        raise ValueError(f"Uncertainty too large (avg_sd={avg_sd:.3f})")
    
    return True

# Test 7: Configuration validation
def test_configuration():
    # Check that configuration parameters are reasonable
    if not (0 < CFG.window_frac <= 1):
        raise ValueError(f"Invalid window_frac: {CFG.window_frac}")
    
    if not (0 < CFG.ema_alpha <= 1):
        raise ValueError(f"Invalid ema_alpha: {CFG.ema_alpha}")
    
    if CFG.savgol_poly < 1:
        raise ValueError(f"Invalid savgol_poly: {CFG.savgol_poly}")
    
    return True

# Run all tests
print("Running comprehensive validation tests...\n")

test_results = []
test_results.append(test_section("Imports and Setup", test_imports))
test_results.append(test_section("Data Structure", test_data_structure))
test_results.append(test_section("Sentiment Scores", test_sentiment_scores))
test_results.append(test_section("Smoothing Algorithms", test_smoothing))
test_results.append(test_section("Peak Detection", test_peak_detection))
test_results.append(test_section("Bootstrap Analysis", test_bootstrap))
test_results.append(test_section("Configuration", test_configuration))

# Summary
passed = sum(test_results)
total = len(test_results)

print("\n" + "=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
print(f"Tests passed: {passed}/{total}")

if passed == total:
    print("🎉 ALL TESTS PASSED! The notebook is working correctly.")
else:
    print("⚠️  Some tests failed. Please check the issues above.")
    
print(f"\nDataset summary:")
print(f"- Sentences processed: {len(df)}")
print(f"- Sentiment range: [{df['mean_z'].min():.3f}, {df['mean_z'].max():.3f}]")
print(f"- Smoothing method: {CFG.smoothing}")
print(f"- Peaks detected: {len(df_peaks)}")

if 'smooth_sd' in df.columns:
    print(f"- Average uncertainty: {df['smooth_sd'].mean():.3f}")

print("\nNotebook validation complete!")